#### Comparison of pure Python and coppertop

The following code snippets solve the M&M problem from Alan Downey's book _Think Bayes_.


<br>

#### Bayes Refresher

See "An Essay towards solving a Problem in the Doctrine of Chances" - e.g. https://zenodo.org/record/1432246/files/article.pdf.

from PROP 3

$$
\begin{align}
\mathbf{P}\left(B \cap A\right) = \mathbf{P}\left(B\mathbin{\vert}A\right)\cdot \mathbf{P}\left(A\right)\\
\end{align}
$$

and
$$\mathbf{P}(A \cap B) = \mathbf{P}(B \cap A)$$

we have
$$
\begin{align}
\mathbf{P}( A\mathbin{\vert}B) \cdot \mathbf{P}(B)=\mathbf{P}(B\mathbin{\vert}A)\cdot \mathbf{P}(A)
\end{align}
$$

aka
$$
\begin{align}
\mathbf{P}( hypothesis\mathbin{\vert}data) \cdot \mathbf{P}(data)=\mathbf{P}(data\mathbin{\vert}hypothesis)\cdot \mathbf{P}(hypothesis)
\end{align}
$$

<br>

Equivalently (after new data is known) we have the comtemporaneous form, i.e. 
$$
\begin{align}
posterior =likelihood\cdot prior \cdot constant
\end{align}
$$

<br>

#### The M&M Problem

M&M’s are small candy-coated chocolates that come in a variety of colors.
Mars, Inc., which makes M&M’s, changes the mixture of colors from time
to time.

In 1995, they introduced blue M&M’s. Before then, the color mix in a bag
of plain M&M’s was 30% Brown, 20% Yellow, 20% Red, 10% Green, 10%
Orange, 10% Tan. Afterward it was 24% Blue , 20% Green, 16% Orange,
14% Yellow, 13% Red, 13% Brown.

Suppose a friend of mine has two bags of M&M’s, and he tells me that one
is from 1994 and one from 1996. He won’t tell me which is which, but he
gives me one M&M from each bag. One is yellow and one is green. What is
the probability that the yellow one came from the 1994 bag?

<br>

##### Pure Python version

In [1]:
def pmfMul(A, B):
    kps = {}
    for k in A.keys():
        kps[k] = A[k] * B[k]
    return kps

def normalise(kps):
    t = 0
    for p in kps.values():
        t += p
    t = 1 / t
    for k in kps.keys():
        kps[k] *= t
    return kps

In [2]:
bag1994 = normalise(dict(Brown=30, Yellow=20, Red=20, Green=10, Orange=10, Tan=10))
bag1996 = normalise(dict(Brown=13, Yellow=14, Red=13, Green=20, Orange=16, Blue=24))

for e in [bag1994, bag1996]:
    print(e)

{'Brown': 0.3, 'Yellow': 0.2, 'Red': 0.2, 'Green': 0.1, 'Orange': 0.1, 'Tan': 0.1}
{'Brown': 0.13, 'Yellow': 0.14, 'Red': 0.13, 'Green': 0.2, 'Orange': 0.16, 'Blue': 0.24}


<br>

hypA -> yellow is from 1994, green is from 1996\
hypB -> green is from 1994, yellow is from 1996

In [3]:
prior = normalise(dict(hypA=0.5, hypB=0.5))

likelihood = dict(
    hypA=bag1994['Yellow'] * bag1996['Green'], 
    hypB=bag1994['Green'] * bag1996['Yellow']
)

posterior = normalise(pmfMul(prior, likelihood))

print(prior)
print(likelihood)
print(posterior)

{'hypA': 0.5, 'hypB': 0.5}
{'hypA': 0.04000000000000001, 'hypB': 0.014000000000000002}
{'hypA': 0.7407407407407408, 'hypB': 0.25925925925925924}


<br>

##### What's the problem we are trying to solve?

This Python code looks very simple. And thus far it is. However, once we want to start scaling and improving the experience of working with our library we atart to find the code get's messy very quickly.

For a moment consider, formatting the result to a fixed precision, running several priors and inserting debugging info. Below we show how this could be done in coppertop.

<br>

##### Coppertop Version

* create some types, PMF and L, instead of dict
* use the pipe operator

In [5]:
from coppertop.pipe import *

from dm.pp import PP                     # import the PP (prettry print) function which prints the argument and then returns the argument

from bones.lang.metatypes import BTAtom
from dm.core.structs import tvstruct
from dm.core import collect
import bones.lang.metatypes

bones.lang.metatypes.REPL_OVERRIDE_MODE = True     # allow ideally immutable state to be updated in a repl

In [6]:
# create new types for a collection of (key, probability) pairs and a struct containing (key, probability) pairs
keyProbs = BTAtom.ensure('keyProbs')
keyProbsStruct = (keyProbs & tvstruct).setConstructor(tvstruct)       # the '&' creates a type that is the intersection of keyProbs and tvstruct

# create new types for PMF (probability mass function) and L (likelihood)
PMF = (BTAtom.ensure('_PMF') & keyProbsStruct).nameAs('PMF')        # _PMF is the atomic type we use to indicate things that are like a probability mass function
L = (BTAtom.ensure('_L') & keyProbsStruct).nameAs('L')              # similarly for _L

def _newPmf(*args, **kwargs):
    kps = tvstruct(*args, **kwargs)
    kps._t = PMF
    if kps:
        t = 0
        for p in kps._values():
            t += p
        t = 1 / t
        for k in kps._keys():
            kps[k] *= t
    return kps

PMF.setConstructor(_newPmf)
L.setConstructor(keyProbsStruct)

@coppertop(style=binary)
def pmfMul(lhs:keyProbsStruct, rhs:keyProbsStruct) -> keyProbsStruct:
    answer = keyProbsStruct()
    for k in lhs._keys():
        answer[k] = lhs[k] * rhs[k]
    return answer

@coppertop
def normalise(kps:keyProbsStruct) -> PMF:
    return _newPmf(kps)


In [7]:
bag1994 = PMF(Brown=30, Yellow=20, Red=20, Green=10, Orange=10, Tan=10) >> PP
bag1996 = PMF(Brown=13, Yellow=14, Red=13, Green=20, Orange=16, Blue=24) >> PP

PMF(Brown=0.3, Yellow=0.2, Red=0.2, Green=0.1, Orange=0.1, Tan=0.1)
PMF(Brown=0.13, Yellow=0.14, Red=0.13, Green=0.2, Orange=0.16, Blue=0.24)


<br>

hypA -> yellow is from 1994, green is from 1996\
hypB -> green is from 1994, yellow is from 1996

In [12]:
prior = PMF(hypA=0.5, hypB=0.5)

likelihood = L(
    hypA=bag1994.Yellow * bag1996.Green, 
    hypB=bag1994.Green * bag1996.Yellow
)

posterior = prior >> pmfMul >> likelihood >> normalise

prior >> PP
likelihood >> PP
posterior >> PP

PMF(hypA=0.500, hypB=0.500)
L(hypA=0.040, hypB=0.014)
PMF(hypA=0.741, hypB=0.259)


PMF(hypA=0.7407407407407408, hypB=0.25925925925925924)

In [13]:
(prior, likelihood, posterior) >> collect >> typeOf   # display their types

[PMF, L, PMF]

<br>

##### Improved display

We can tidy up the output a little by overloading PP to display the probabilities to three decimal places

In [14]:
from dm.pp import formatStruct

formatPmf = formatStruct(_, 'PMF', '.3f', '.3f', ', ')
formatL = formatStruct(_, 'L', '.3f', '.3f', ', ')

@coppertop
def PP(x:L) -> L:
    print(x >> formatL)
    return x

@coppertop
def PP(x:PMF):
    print(x >> formatPmf)
    return x

In [15]:
bag1994 = PMF(Brown=30, Yellow=20, Red=20, Green=10, Orange=10, Tan=10) >> PP
bag1996 = PMF(Brown=13, Yellow=14, Red=13, Green=20, Orange=16, Blue=24) >> PP

PMF(Brown=0.300, Yellow=0.200, Red=0.200, Green=0.100, Orange=0.100, Tan=0.100)
PMF(Brown=0.130, Yellow=0.140, Red=0.130, Green=0.200, Orange=0.160, Blue=0.240)


In [16]:
prior = PMF(hypA=0.5, hypB=0.5)

likelihood = L(
    hypA=bag1994.Yellow * bag1996.Green, 
    hypB=bag1994.Green * bag1996.Yellow
)

posterior = prior  >> PP >> pmfMul >> (likelihood  >> PP) >> normalise >> PP

PMF(hypA=0.500, hypB=0.500)
L(hypA=0.040, hypB=0.014)
PMF(hypA=0.741, hypB=0.259)


In [22]:
"case 1" >> PP
PMF(hypA=0.25, hypB=0.75) >> PP >> pmfMul >> (likelihood >> PP) >> normalise >> PP
"\ncase 2" >> PP
PMF(hypA=0.5, hypB=0.5) >> PP >> pmfMul >> (likelihood >> PP) >> normalise >> PP
"\ncase 3" >> PP
PMF(hypA=0.75, hypB=0.25)  >> PP >> pmfMul >> (likelihood >> PP) >> normalise >> PP;


case 1
PMF(hypA=0.250, hypB=0.750)
L(hypA=0.040, hypB=0.014)
PMF(hypA=0.488, hypB=0.512)

case 2
PMF(hypA=0.500, hypB=0.500)
L(hypA=0.040, hypB=0.014)
PMF(hypA=0.741, hypB=0.259)

case 3
PMF(hypA=0.750, hypB=0.250)
L(hypA=0.040, hypB=0.014)
PMF(hypA=0.896, hypB=0.104)
